In [1]:
# Importing modules
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Loading ESOL data
esol_data_raw = pd.read_csv("data/ESOL.csv")[["smiles", "measured log solubility in mols per litre"]]
esol_data_raw = esol_data_raw.rename(columns={"measured log solubility in mols per litre":"target"})

# Loading Lipophilicity data
lipophil_data_raw = pd.read_csv("data/Lipophilicity.csv")[["smiles", "exp"]]
lipophil_data_raw = lipophil_data_raw.rename(columns={"exp":"target"})

# Loading retention time (RT) data
rt_data_raw = pd.read_csv("data/RT.csv")
rt_data_raw = rt_data_raw.rename(columns={"SMILES":"smiles","RTs":"target"})

# Loading B3DB data
b3db_data_raw = pd.read_csv("data/B3DB.tsv", sep="\t")[["SMILES", "logBB"]]
b3db_data_raw = b3db_data_raw.rename(columns={"SMILES":"smiles", "logBB":"target"})

# Loading FreeSolv data
freesolv_data_raw = pd.read_csv("data/FreeSolv.csv")[["smiles", "expt"]]
freesolv_data_raw = freesolv_data_raw.rename(columns={"SMILES":"smiles", "expt":"target"})

In [3]:
summary_data = {
    "ESOL": len(esol_data_raw), 
    "FreeSolv": len(freesolv_data_raw), 
    "Lipophilicity": len(lipophil_data_raw), 
    "B3DB": len(b3db_data_raw), 
    "RT": len(rt_data_raw)
}

df_summary = pd.DataFrame(list(summary_data.items()), columns=["Dataset", "Count"]).sort_values("Count", ascending=False)
palette = {"ESOL": "orange", "FreeSolv": "blue", "Lipophilicity": "red", "B3DB": "purple", "RT": "green"}

fig, axes = plt.subplots(2, 3, dpi=100, figsize=(15, 10))
point_kws = dict(color="black", alpha=0.4, size=3)

def plot_single_violin(ax, data, title, color):
    sns.violinplot(y=data["target"], ax=ax, color=color, linecolor="black", alpha=0.8)
    sns.stripplot(y=data["target"], ax=ax, **point_kws)
    ax.set_xlabel(f"{title}", fontweight='bold')
    ax.set_ylabel("Target Value", fontweight='bold')
    sns.despine(ax=ax)

plot_single_violin(axes[0, 0], esol_data_raw, "ESOL", "orange")
plot_single_violin(axes[0, 1], freesolv_data_raw, "FreeSolv", "blue")
plot_single_violin(axes[0, 2], lipophil_data_raw, "Lipophilicity", "red")
plot_single_violin(axes[1, 0], b3db_data_raw, "B3DB", "purple")
plot_single_violin(axes[1, 1], rt_data_raw, "RT", "green")

ax_bar = axes[1, 2]
sns.barplot(
    x="Dataset", 
    y="Count", 
    data=df_summary, 
    hue="Dataset", 
    palette=palette, 
    alpha=0.8,
    width=0.9,      
    dodge=False,    
    edgecolor="black",
    ax=ax_bar
)

for p in ax_bar.patches:
    if p.get_height() > 0:
        ax_bar.annotate(f'{int(p.get_height())}', 
                        (p.get_x() + p.get_width() / 2., p.get_height()), 
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 5),
                        textcoords='offset points')

ax_bar.set_ylabel("Number of SMILES", fontweight='bold')
ax_bar.set_xlabel("")
ax_bar.tick_params(axis='x', rotation=45)
ax_bar.set_xticklabels(ax_bar.get_xticklabels(), weight='bold')
ax_bar.set_ylim(0, df_summary['Count'].max() * 1.15)

if ax_bar.get_legend():
    ax_bar.get_legend().remove()

sns.despine(ax=ax_bar)

plt.tight_layout()
plt.savefig("results/EDA_Plot.png", dpi=300)
plt.close()